# S5. AI 윤리와 공정성 (온라인 보충)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/online/%EC%98%A8%EB%9D%BC%EC%9D%B8%EB%B3%B4%EC%B6%A9_S5_AI_%EC%9C%A4%EB%A6%AC%EC%99%80_%EA%B3%B5%EC%A0%95%EC%84%B1.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 실습: 공정성 측정과 적대적 공격·방어

### 🔬 실습 — 공정성 지표 측정과 FGSM 공격$$적대적 훈련

① **공정성 지표 측정**: 집단별 양성률$$TPR로 DP$$EO 격차 계산.


In [ ]:
import torch
def fairness_metrics(y_true, y_pred, group):
    out = {}
    for g in group.unique():
        m = (group == g)
        ppr = y_pred[m].float().mean()                       # 양성 판정률
        pos = m & (y_true == 1)
        tpr = (y_pred[pos] == 1).float().mean() if pos.any() else torch.tensor(0.)
        out[int(g)] = (ppr.item(), tpr.item())
    dp = abs(out[0][0] - out[1][0])                          # Demographic Parity 격차
    eo = abs(out[0][1] - out[1][1])                          # Equal Opportunity 격차
    return out, dp, eo

② **FGSM 공격**: 손실 기울기 부호로 적대적 예제 생성.


In [ ]:
import torch.nn.functional as F
def fgsm(model, x, y, eps=0.03):
    x = x.clone().detach().requires_grad_(True)
    loss = F.cross_entropy(model(x), y)
    loss.backward()
    x_adv = x + eps * x.grad.sign()              # 부호 방향 섭동
    return x_adv.clamp(0, 1).detach()

③ **적대적 훈련**: 매 배치에서 적대적 예제를 만들어 함께 학습.


In [ ]:
# 적대적 훈련 시연용 소형 모델과 합성 데이터
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

model = nn.Sequential(nn.Linear(20, 32), nn.ReLU(), nn.Linear(32, 2)).to(device)
X_syn = torch.rand(128, 20)                  # fgsm의 clamp(0,1)에 맞춰 [0,1] 범위
y_syn = torch.randint(0, 2, (128,))
train_loader = DataLoader(TensorDataset(X_syn, y_syn), batch_size=32, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for x, y in train_loader:
    x, y = x.to(device), y.to(device)
    x_adv = fgsm(model, x, y, eps=0.03)          # 적대적 예제 생성
    logits = model(x_adv)                        # 적대적 예제로 학습
    loss = F.cross_entropy(logits, y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
